In [ ]:
!pip install biopython pandas numpy scikit-learn seaborn

import pandas as pd
import numpy as np
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio import SeqIO
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter


In [ ]:
# ================================
# 0. Constant Definitions
# ================================

AAS = "ACDEFGHIKLMNPQRSTVWY"
SW_WINDOWS = [20, 40]

AA_GROUPS = {
    "KR": "KR",
    "KRH": "KRH",
    "ED": "ED",
}

AA_GROUPS_EXT = {
    "STNQCH": "STNQCH",
    "ILMV":   "ILMV",
    "FWY":    "FWY",
}

# Merge both group sets for AAC computation
AAC_GROUPS_ALL = {
    **AA_GROUPS_EXT,
    **AA_GROUPS
}

def clean_seq(seq):
    return seq.replace("U", "C")

# ================================
# 1. Load High / Mid / Low Group IDs
# ================================

df_high = pd.read_excel("High_group_ID.xlsx")
df_mid  = pd.read_excel("Mid_group_ID.xlsx")
df_low  = pd.read_excel("Low_group_ID.xlsx")

df_high["Class3"] = "High"
df_mid["Class3"]  = "Mid"
df_low["Class3"]  = "Low"

df_class = pd.concat([df_high, df_mid, df_low], ignore_index=True)

# ================================
# 2. Sequence Processing
# ================================

seqs = {rec.id: str(rec.seq) for rec in SeqIO.parse(
    "ALL_protein_sequences.fasta", "fasta")}

df_seq = pd.DataFrame(seqs.items(), columns=["ID", "Sequence"])

# Merge sequence data directly
df = df_class.merge(df_seq, on="ID")
df["Sequence"] = df["Sequence"].map(clean_seq)

# ================================
# 3. Binary Labeling (High / Low)
# ================================

df_binary = df[df["Class3"].isin(["High", "Low"])].copy()
df_binary["Class2"] = df_binary["Class3"].map({"Low": 0, "High": 1})

# ================================
# 4. Physicochemical Features + AAC (+ Group AAC)
# ================================

def physchem_features(seq):
    prot = ProteinAnalysis(seq)
    aa = prot.count_amino_acids()
    L = len(seq)

    # --- Basic Physicochemical Properties ---
    base = [
        prot.molecular_weight(),
        prot.gravy(),
        prot.charge_at_pH(7.5),
        prot.charge_at_pH(6.8),
        prot.charge_at_pH(6.8) - prot.charge_at_pH(7.5),
        prot.isoelectric_point()
    ]

    # --- Single Amino Acid Composition (AAC) ---
    aac = [aa[a] / L for a in AAS]

    # --- Grouped Amino Acid Composition (EXT + Charge) ---
    aac_group = [
        sum(aa[a] for a in group) / L
        for group in AAC_GROUPS_ALL.values()
    ]

    return base + aac + aac_group


pc_cols = (
    ["MW","GRAVY","Charge_7.5","Charge_6.8","DeltaCharge_6.8_7.5","pI"]
    + [f"AAC_{a}" for a in AAS]
    + [f"AACgrp_{k}" for k in AAC_GROUPS_ALL]
)

df_pc_all = pd.DataFrame(
    [physchem_features(s) for s in df["Sequence"]],
    columns=pc_cols
)

# ================================
# 5. Terminal Bias Index
# ================================

def termial_bias(seq, aa_set):
    L = len(seq)
    positions = [
        abs((i + 0.5) / L - 0.5)
        for i, a in enumerate(seq)
        if a in aa_set
    ]
    return np.mean(positions) if positions else 0.0

df_term_aa = pd.DataFrame(
    [
        [termial_bias(s, a) for a in AAS]
        for s in df["Sequence"]
    ],
    columns=[f"TermBias_{a}" for a in AAS]
)

df_term_group = pd.DataFrame(
    [
        [termial_bias(s, g) for g in AA_GROUPS_EXT.values()]
        for s in df["Sequence"]
    ],
    columns=[f"TermBias_{k}" for k in AA_GROUPS_EXT]
)

df_term_charge = pd.DataFrame(
    [
        [termial_bias(s, g) for g in AA_GROUPS.values()]
        for s in df["Sequence"]
    ],
    columns=[f"TermBias_{k}" for k in AA_GROUPS]
)

# ================================
# 6. Sliding Window Features (Theoretical Variance Normalization)
# ================================

def sliding_window_var_norm_aa(seq, aas):
    out = []
    L = len(seq)
    cnt_all = Counter(seq)

    for w in SW_WINDOWS:
        for a in aas:
            p = cnt_all[a] / L
            vals = []

            for i in range(L - w + 1):
                sub = seq[i:i+w]
                vals.append(sub.count(a) / w)

            var_obs = np.var(vals) if vals else 0.0
            var_exp = p * (1 - p) / w
            out.append(var_obs / (var_exp + 1e-6))
    return out

df_sw_aa = pd.DataFrame(
    [sliding_window_var_norm_aa(s, AAS) for s in df["Sequence"]],
    columns=[
        f"SW{w}_varNorm_{a}"
        for w in SW_WINDOWS
        for a in AAS
    ]
)

def sliding_window_var_norm_group(seq, groups):
    out = []
    L = len(seq)
    cnt_all = Counter(seq)

    for w in SW_WINDOWS:
        for k, g in groups.items():
            p = sum(cnt_all[a] for a in g) / L
            vals = []

            for i in range(L - w + 1):
                sub = seq[i:i+w]
                vals.append(sum(sub.count(a) for a in g) / w)

            var_obs = np.var(vals) if vals else 0.0
            var_exp = p * (1 - p) / w
            out.append(var_obs / (var_exp + 1e-6))
    return out

df_sw_group = pd.DataFrame(
    [sliding_window_var_norm_group(s, AA_GROUPS_EXT) for s in df["Sequence"]],
    columns=[
        f"SW{w}_varNorm_{k}"
        for w in SW_WINDOWS
        for k in AA_GROUPS_EXT
    ]
)

df_sw_charge = pd.DataFrame(
    [sliding_window_var_norm_group(s, AA_GROUPS) for s in df["Sequence"]],
    columns=[
        f"SW{w}_varNorm_{k}"
        for w in SW_WINDOWS
        for k in AA_GROUPS
    ]
)

# ================================
# 7. k-mer (k=2): Symmetrical Pairing & Mean Frequency Filtering
# ================================

KMER_K = 2

# Combine AB and BA pairs symmetrically (e.g., AA, AC, AD, ..., YY)
KMER2_SYM_LIST = sorted(
    set("".join(sorted(a + b)) for a in AAS for b in AAS)
)

def kmer2_symmetric_features(seq):
    L = len(seq)

    counts = Counter(
        "".join(sorted(seq[i:i+2]))
        for i in range(L - 1)
    )

    denom = max(L - 1, 1)
    return [counts.get(km, 0) / denom for km in KMER2_SYM_LIST]

df_kmer2_sym = pd.DataFrame(
    [kmer2_symmetric_features(s) for s in df["Sequence"]],
    columns=[f"KMER2sym_{km}" for km in KMER2_SYM_LIST]
)

# Filter by mean frequency across all sequences
KMER_MEAN_FREQ_TH = 1e-3  # Recommended threshold: 1e-3

kmer_mean_freq = df_kmer2_sym.mean()
keep_kmers = kmer_mean_freq[kmer_mean_freq >= KMER_MEAN_FREQ_TH].index

df_kmer2_sym_filt = df_kmer2_sym[keep_kmers]

# ================================
# 8. Final Feature Matrix
# ================================

df_features_all = pd.concat([
    df_pc_all,
    df_term_aa,
    df_term_group,
    df_term_charge,
    df_sw_aa,
    df_sw_group,
    df_sw_charge,
    df_kmer2_sym_filt
], axis=1)

In [ ]:
import random

# ================================
# Model Training for Design Features
# ================================

# Train Random Forest on all available sequence-derived features
X_design = df_features_all.loc[df_binary.index]
y_design = df_binary["Class2"]

clf_design = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
clf_design.fit(X_design, y_design)

imp = pd.Series(
    clf_design.feature_importances_,
    index=df_features_all.columns
).sort_values(ascending=False)

# ================================
# Feature Selection: Top 100 Features
# ================================

TOPN = 100
TOP_FEATURES = imp.head(TOPN).index.tolist()
print("Number of top features selected for design:", len(TOP_FEATURES))

# --- Retrain Random Forest using only Top 100 features ---
X_design_top = X_design[TOP_FEATURES]

clf_design_top = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
clf_design_top.fit(X_design_top, y_design)

# ================================
# Compute Distribution Penalties (Mean & Std Dev for Top 100)
# ================================

df_design = df_features_all.loc[y_design.index]
df_design_top = df_design[TOP_FEATURES]

mu_high = df_design_top.loc[y_design == 1].mean()
std_high = df_design_top.loc[y_design == 1].std()

mu_low  = df_design_top.loc[y_design == 0].mean()
std_low = df_design_top.loc[y_design == 0].std()

# Avoid division by zero
std_high = std_high.replace(0, 1e-6)
std_low  = std_low.replace(0, 1e-6)

def distribution_penalty(X_df, mu, std, weight=0.5):
    z = (X_df[TOP_FEATURES].iloc[0] - mu) / std
    return weight * np.mean(z ** 2)

# ================================
# Feature Extraction Function for Sequence Design
# ================================

def extract_features_design(seq):
    seq = clean_seq(seq)
    prot = ProteinAnalysis(seq)
    aa = prot.count_amino_acids()
    L = len(seq)

    feat_dict = {}

    # --- Physicochemical Properties ---
    feat_dict["MW"] = prot.molecular_weight()
    feat_dict["GRAVY"] = prot.gravy()
    feat_dict["Charge_7.5"] = prot.charge_at_pH(7.5)
    feat_dict["Charge_6.8"] = prot.charge_at_pH(6.8)
    feat_dict["DeltaCharge_6.8_7.5"] = (
        prot.charge_at_pH(6.8) - prot.charge_at_pH(7.5)
    )
    feat_dict["pI"] = prot.isoelectric_point()

    # --- Amino Acid Composition (AAC) ---
    for a in AAS:
        feat_dict[f"AAC_{a}"] = aa[a] / L

    # --- Grouped AAC ---
    for k, g in AAC_GROUPS_ALL.items():
        feat_dict[f"AACgrp_{k}"] = sum(aa[a] for a in g) / L

    # --- Terminal Bias Indices ---
    for a in AAS:
        feat_dict[f"TermBias_{a}"] = termial_bias(seq, a)

    for k, g in AA_GROUPS_EXT.items():
        feat_dict[f"TermBias_{k}"] = termial_bias(seq, g)

    for k, g in AA_GROUPS.items():
        feat_dict[f"TermBias_{k}"] = termial_bias(seq, g)

    # --- Sliding Window (Single AA) ---
    sw_aa = sliding_window_var_norm_aa(seq, AAS)
    idx = 0
    for w in SW_WINDOWS:
        for a in AAS:
            feat_dict[f"SW{w}_varNorm_{a}"] = sw_aa[idx]
            idx += 1

    # --- Sliding Window (Groups EXT) ---
    sw_grp = sliding_window_var_norm_group(seq, AA_GROUPS_EXT)
    idx = 0
    for w in SW_WINDOWS:
        for k in AA_GROUPS_EXT:
            feat_dict[f"SW{w}_varNorm_{k}"] = sw_grp[idx]
            idx += 1

    # --- Sliding Window (Charge Groups) ---
    sw_chg = sliding_window_var_norm_group(seq, AA_GROUPS)
    idx = 0
    for w in SW_WINDOWS:
        for k in AA_GROUPS:
            feat_dict[f"SW{w}_varNorm_{k}"] = sw_chg[idx]
            idx += 1

    # --- k-mer (Symmetric, Filtered) ---
    kmer_vals = kmer2_symmetric_features(seq)
    for km, v in zip(KMER2_SYM_LIST, kmer_vals):
        col = f"KMER2sym_{km}"
        if col in df_kmer2_sym_filt.columns:
            feat_dict[col] = v

    # --- Align features to match TOP_FEATURES ordering ---
    feats = [feat_dict[c] for c in TOP_FEATURES]

    return np.array(feats).reshape(1, -1)

def score_high(seq, gamma=0.5):
    X = extract_features_design(seq)
    rf_score = clf_design_top.predict_proba(X)[0, 1]

    X_df = pd.DataFrame(X, columns=TOP_FEATURES)
    p_dist = distribution_penalty(X_df, mu_high, std_high, weight=gamma)

    return rf_score - p_dist


def score_low(seq, gamma=0.5):
    X = extract_features_design(seq)
    rf_score = clf_design_top.predict_proba(X)[0, 0]

    X_df = pd.DataFrame(X, columns=TOP_FEATURES)
    p_dist = distribution_penalty(X_df, mu_low, std_low, weight=gamma)

    return rf_score - p_dist

# ================================
# Sequence Optimization Algorithms
# ================================

ALL_AA = list(AAS)

def mutate_global(seq, p=0.03):
    """Global mutation: Randomly changes amino acids with probability p."""
    seq = list(seq)
    for i in range(len(seq)):
        if random.random() < p:
            seq[i] = random.choice(ALL_AA)
    return "".join(seq)

def mutate_local(seq, n_swap=2):
    """Local mutation: Swaps positions of amino acids within the sequence."""
    seq = list(seq)
    L = len(seq)

    for _ in range(n_swap):
        i, j = random.sample(range(L), 2)
        seq[i], seq[j] = seq[j], seq[i]

    return "".join(seq)

def optimize_sequence_two_stage(
    init_seq,
    score_func,
    steps_global=1000,
    steps_local=1000
):
    """Two-stage optimization: 1) Composition search -> 2) Patterning refinement"""
    best_seq = init_seq
    best_score = score_func(best_seq)

    # --- Stage 1: Composition search ---
    for _ in range(steps_global):
        new = mutate_global(best_seq, p=0.03)
        s = score_func(new)
        if s > best_score:
            best_seq, best_score = new, s

    # --- Stage 2: Patterning refinement ---
    for _ in range(steps_local):
        new = mutate_local(best_seq, n_swap=2)
        s = score_func(new)
        if s > best_score:
            best_seq, best_score = new, s

    return best_seq, best_score

# ================================
# Initial Sequence Generation (Based on Mean AAC)
# ================================

aac_cols = [c for c in df_features_all.columns if c.startswith("AAC_")]
aac_mean = df_features_all[aac_cols].mean().values
aas = [c.replace("AAC_", "") for c in aac_cols]

def random_sequence_from_aac(L=300):
    """Generates a random sequence sampling amino acids weighted by dataset AAC mean."""
    return "".join(
        random.choices(aas, weights=aac_mean, k=1)[0]
        for _ in range(L)
    )

In [ ]:
def generate_sequences(score_func, n=1, L=300):
    out = []
    for i in range(n):
        init = random_sequence_from_aac(L)
        seq, score = optimize_sequence_two_stage(init, score_func)
        out.append((f"seq_{i:03d}", seq, score))
    return out

high_seqs = generate_sequences(score_high)
low_seqs  = generate_sequences(score_low)

def write_fasta(seqs, filename):
    with open(filename, "w") as f:
        for name, seq, score in seqs:
            f.write(f">{name}|score={score:.3f}\n{seq}\n")


write_fasta(high_seqs, "Designed_High_seq.fasta")
write_fasta(low_seqs,  "Designed_Low_seq.fasta")